# MRI Volume Preprocessing & Metrics Pipeline
### Yugma TechFest 2.0 — MedhaDrishti AI Hackathon (Stage 1 & Stage 2)

This notebook performs, for **Brain MRI (T1, T1c, T2, FLAIR)** and **Spine MRI (T1, T1c, T2, STIR)**,
normal + pathological volumes in NIfTI (`.nii` / `.nii.gz`) format:

1. **Dataset discovery** — scans the given folder structure and builds an index of every volume.
2. **Stage 1 — Image property assessment (RAW)**: Mean, Std, Contrast, Sharpness, Edge strength,
   Complexity, Entropy, Noise level.
3. **Stage 2 — Preprocessing**: denoising, bias-field correction, intensity normalization,
   CLAHE-based contrast enhancement, background/non-significant region masking, resizing.
4. **Stage 2 — Image property assessment (PREPROCESSED)** — same stats as step 2, recomputed.
5. **Reference-based quality metrics (RAW vs PREPROCESSED)**: PSNR, SSIM, MSE, RMSE, UQI, FSIM,
   GMSD, VIF, and (optionally) LPIPS.
6. **No-reference perceptual metrics** on preprocessed volumes: BRISQUE, NIQE, PIQE (optional, if
   the corresponding libraries are installed).
7. Saves preprocessed volumes to disk and **writes one consolidated CSV** with all metrics
   (`mri_preprocessing_metrics.csv`) — one row per (patient, modality, stage).

> **Compute:** This notebook is CPU-only. No GPU is required — everything here is classical
> image processing / statistics, not deep learning training. GPU becomes relevant later in
> Stage 3 (AI-based enhancement) and Stage 4 (segmentation).

---
## 0. Setup — install & import dependencies


In [ ]:

# Core (required)
import sys, subprocess

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                     "--break-system-packages", pkg])

required = ["nibabel", "numpy", "scipy", "scikit-image", "opencv-python-headless",
            "pandas", "matplotlib", "tqdm"]
for p in required:
    pip_install(p)

# Optional (nice-to-have; notebook degrades gracefully if these fail to install)
optional = ["SimpleITK", "sewar", "piq", "torch", "lpips"]
OPTIONAL_STATUS = {}
for p in optional:
    try:
        pip_install(p)
        OPTIONAL_STATUS[p] = "installed"
    except Exception as e:
        OPTIONAL_STATUS[p] = f"failed: {e}"

print("Optional package status:", OPTIONAL_STATUS)


In [ ]:

import os
import re
import glob
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import nibabel as nib
import cv2
import matplotlib.pyplot as plt

from scipy import ndimage as ndi
from scipy.stats import entropy as scipy_entropy
from skimage.restoration import denoise_nl_means, estimate_sigma
from skimage.filters import threshold_otsu, sobel
from skimage.metrics import (
    peak_signal_noise_ratio as sk_psnr,
    structural_similarity as sk_ssim,
    mean_squared_error as sk_mse,
)
from skimage.exposure import equalize_adapthist
from tqdm.auto import tqdm

# Optional imports (used only if available)
try:
    import SimpleITK as sitk
    HAVE_SITK = True
except Exception:
    HAVE_SITK = False

try:
    from sewar.full_ref import uqi as sewar_uqi, fsim as sewar_fsim, \
        vifp as sewar_vif
    HAVE_SEWAR = True
except Exception:
    HAVE_SEWAR = False

try:
    import piq
    import torch
    HAVE_PIQ = True
except Exception:
    HAVE_PIQ = False

print(f"SimpleITK available: {HAVE_SITK}")
print(f"sewar available (UQI/FSIM/VIF): {HAVE_SEWAR}")
print(f"piq available (BRISQUE/NIQE/LPIPS proxy): {HAVE_PIQ}")


## 1. Configuration — set your dataset & output paths

Point `DATASET_ROOT` to the folder(s) that hold the Hackathon dataset.
Expected structure (as per problem statement):

```
DATASET_ROOT/
  Brain/
    Normal/<patient_id>/{T1,T1c,T2,FLAIR}.nii.gz
    Pathological/<patient_id>/{T1,T1c,T2,FLAIR}.nii.gz
  Spine/
    Normal/<patient_id>/{T1,T1c,T2,STIR}.nii.gz
    Pathological/<patient_id>/{T1,T1c,T2,STIR}.nii.gz
```

If your folders are named/organized slightly differently, edit `discover_volumes()` in
Section 2 — it uses simple keyword matching on filenames so minor naming differences
(e.g. `T1c` vs `T1CE`, `Flair` vs `FLAIR`) are still picked up.


In [ ]:

# ==== EDIT THESE PATHS ====
DATASET_ROOT   = "/mnt/user-data/uploads/MRI_Dataset"     # <- root folder of the hackathon dataset
OUTPUT_ROOT    = "/mnt/user-data/outputs/preprocessed_MRI" # <- where enhanced volumes are saved
CSV_OUTPUT     = "/mnt/user-data/outputs/mri_preprocessing_metrics.csv"
TARGET_SHAPE   = (128, 128, 128)   # resampled volume shape (D,H,W) used for metric computation
MAX_VOLUMES    = None               # set an int to limit volumes while testing, else None
# ===========================

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(os.path.dirname(CSV_OUTPUT), exist_ok=True)


## 2. Dataset discovery

Walks the dataset root and builds a table of every volume found, tagged with:
`region` (Brain/Spine), `condition` (Normal/Pathological), `patient_id`, `modality`, `filepath`.


In [ ]:

MODALITY_PATTERNS = {
    "T1":    [r"(?<![a-z])t1(?!c)(?!ce)"],
    "T1c":   [r"t1c", r"t1ce", r"t1_ce", r"t1-contrast", r"contrast[_-]?t1"],
    "T2":    [r"(?<![a-z])t2(?!\*)"],
    "FLAIR": [r"flair"],
    "STIR":  [r"stir", r"spair"],
}

def classify_modality(filename):
    fname = filename.lower()
    for modality, patterns in MODALITY_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, fname):
                return modality
    return "UNKNOWN"

def discover_volumes(root):
    records = []
    nii_files = glob.glob(os.path.join(root, "**", "*.nii"), recursive=True) + \
                glob.glob(os.path.join(root, "**", "*.nii.gz"), recursive=True)

    for fp in nii_files:
        rel = os.path.relpath(fp, root)
        parts = rel.split(os.sep)
        lower_parts = [p.lower() for p in parts]

        region = "Brain" if any("brain" in p for p in lower_parts) else \
                 ("Spine" if any(("spine" in p) or ("lumbar" in p) or ("ls" == p) for p in lower_parts) else "Unknown")

        condition = "Pathological" if any(("patho" in p) or ("tumor" in p) or ("tumour" in p) or ("disease" in p) for p in lower_parts) \
                    else ("Normal" if any("normal" in p for p in lower_parts) else "Unknown")

        # patient id: assume the folder directly containing the file (or its parent) is the patient id
        patient_id = parts[-2] if len(parts) >= 2 else "unknown_patient"

        modality = classify_modality(os.path.basename(fp))

        records.append({
            "filepath": fp,
            "region": region,
            "condition": condition,
            "patient_id": patient_id,
            "modality": modality,
        })
    return pd.DataFrame(records)

if os.path.isdir(DATASET_ROOT):
    df_index = discover_volumes(DATASET_ROOT)
    if MAX_VOLUMES:
        df_index = df_index.head(MAX_VOLUMES)
    print(f"Found {len(df_index)} volumes.")
    display(df_index.head(20))
else:
    print(f"DATASET_ROOT not found at '{DATASET_ROOT}'. "
          f"Update the path in Section 1, then re-run this cell.")
    df_index = pd.DataFrame(columns=["filepath","region","condition","patient_id","modality"])


In [ ]:

if len(df_index):
    print(df_index.groupby(["region", "condition", "modality"]).size())


## 3. Core I/O and volume utilities

In [ ]:

def load_nii(filepath):
    \"\"\"Load a NIfTI volume, return (data as float32 numpy array, affine, header).\"\"\"
    img = nib.load(filepath)
    data = img.get_fdata().astype(np.float32)
    return data, img.affine, img.header

def save_nii(data, affine, header, out_path):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    new_img = nib.Nifti1Image(data.astype(np.float32), affine, header)
    nib.save(new_img, out_path)

def resample_volume(vol, target_shape=TARGET_SHAPE):
    \"\"\"Resize a 3D volume to target_shape using zoom (order=1, trilinear).\"\"\"
    factors = [t / s for t, s in zip(target_shape, vol.shape)]
    return ndi.zoom(vol, factors, order=1)

def normalize_01(vol):
    vmin, vmax = np.percentile(vol, 0.5), np.percentile(vol, 99.5)
    vol = np.clip(vol, vmin, vmax)
    if vmax - vmin < 1e-8:
        return np.zeros_like(vol)
    return (vol - vmin) / (vmax - vmin)

def foreground_mask(vol):
    \"\"\"Simple Otsu-based mask to separate anatomy from background (defocus non-significant regions).\"\"\"
    flat = vol[vol > 0]
    if flat.size == 0:
        return np.ones_like(vol, dtype=bool)
    try:
        thresh = threshold_otsu(flat)
    except Exception:
        thresh = flat.mean()
    return vol > thresh * 0.15  # lenient threshold to keep anatomical soft tissue


## 4. Preprocessing pipeline (Stage 2)

Steps: bias-field correction (if SimpleITK available) → denoising (Non-local means, slice-wise)
→ intensity normalization → CLAHE contrast enhancement (slice-wise) → background defocus mask
→ resample to common shape.


In [ ]:

def n4_bias_correction(vol):
    \"\"\"Optional N4 bias field correction using SimpleITK. Falls back to identity if unavailable.\"\"\"
    if not HAVE_SITK:
        return vol
    try:
        sitk_img = sitk.GetImageFromArray(vol.astype(np.float32))
        mask_img = sitk.OtsuThreshold(sitk_img, 0, 1, 200)
        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        corrector.SetMaximumNumberOfIterations([20, 20, 10])
        corrected = corrector.Execute(sitk_img, mask_img)
        return sitk.GetArrayFromImage(corrected)
    except Exception as e:
        print("N4 correction skipped:", e)
        return vol

def denoise_volume(vol):
    \"\"\"Slice-wise Non-local means denoising on the normalized volume.\"\"\"
    vol_norm = normalize_01(vol)
    denoised = np.zeros_like(vol_norm)
    for i in range(vol_norm.shape[2]):
        sl = vol_norm[:, :, i]
        sigma_est = np.mean(estimate_sigma(sl)) if sl.max() > 0 else 0
        denoised[:, :, i] = denoise_nl_means(
            sl, h=1.15 * sigma_est if sigma_est > 0 else 0.02,
            fast_mode=True, patch_size=5, patch_distance=6
        )
    return denoised

def clahe_enhance(vol_norm):
    \"\"\"Slice-wise CLAHE (Contrast Limited Adaptive Histogram Equalization).\"\"\"
    enhanced = np.zeros_like(vol_norm)
    for i in range(vol_norm.shape[2]):
        sl = vol_norm[:, :, i]
        if sl.max() <= 0:
            enhanced[:, :, i] = sl
            continue
        sl_uint8 = (sl * 255).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        eq = clahe.apply(sl_uint8)
        enhanced[:, :, i] = eq.astype(np.float32) / 255.0
    return enhanced

def preprocess_volume(vol):
    \"\"\"Full Stage-2 preprocessing pipeline. Input: raw float volume. Output: preprocessed volume (0-1 range).\"\"\"
    vol = n4_bias_correction(vol)
    vol = denoise_volume(vol)          # returns normalized [0,1] + denoised
    vol = clahe_enhance(vol)           # contrast enhancement
    mask = foreground_mask(vol)
    vol = vol * mask                   # defocus non-significant (background) regions
    vol = resample_volume(vol, TARGET_SHAPE)
    vol = np.clip(vol, 0, 1)
    return vol


## 5. Image property metrics (Stage 1 & Stage 2 assessment)

Computed on any single volume: **Mean, Std/Deviation, Contrast (RMS), Sharpness (Laplacian
variance), Edge strength (mean Sobel gradient magnitude), Complexity (mean gradient magnitude),
Entropy, Noise level (background local std).**


In [ ]:

def compute_property_metrics(vol):
    vol_n = normalize_01(vol)
    flat = vol_n.flatten()

    mean_val = float(np.mean(flat))
    std_val  = float(np.std(flat))
    contrast = float(std_val / (mean_val + 1e-8))  # Michelson-like / RMS contrast ratio

    # Sharpness via variance of Laplacian, averaged across slices
    lap_vars = []
    sobel_means = []
    for i in range(vol_n.shape[2]):
        sl = vol_n[:, :, i]
        if sl.max() <= 0:
            continue
        lap = cv2.Laplacian((sl * 255).astype(np.uint8), cv2.CV_64F)
        lap_vars.append(lap.var())
        sobel_means.append(sobel(sl).mean())
    sharpness = float(np.mean(lap_vars)) if lap_vars else 0.0
    edge_strength = float(np.mean(sobel_means)) if sobel_means else 0.0

    # Complexity: mean 3D gradient magnitude
    gx, gy, gz = np.gradient(vol_n)
    complexity = float(np.mean(np.sqrt(gx**2 + gy**2 + gz**2)))

    # Entropy (Shannon, on intensity histogram)
    hist, _ = np.histogram(flat, bins=256, range=(0, 1), density=True)
    hist = hist[hist > 0]
    ent = float(scipy_entropy(hist, base=2))

    # Noise level: std within a low-intensity "background" region (bottom 10th percentile mask)
    bg_thresh = np.percentile(flat, 10)
    bg_vals = flat[flat <= bg_thresh]
    noise_level = float(np.std(bg_vals)) if bg_vals.size else 0.0

    return {
        "mean": mean_val,
        "std_dev": std_val,
        "contrast": contrast,
        "sharpness": sharpness,
        "edge_strength": edge_strength,
        "complexity": complexity,
        "entropy": ent,
        "noise_level": noise_level,
    }


## 6. Reference-based & no-reference quality metrics (RAW vs PREPROCESSED)

PSNR, SSIM, MSE, RMSE always computed (scikit-image). UQI, FSIM, VIF computed if `sewar` is
available. BRISQUE/NIQE/LPIPS computed if `piq`/`torch` are available — otherwise left as `NaN`
so the CSV still has consistent columns.


In [ ]:

def compute_reference_metrics(raw_vol, proc_vol):
    \"\"\"raw_vol and proc_vol must be same shape, range [0,1]. Computed as mean over central slices.\"\"\"
    raw_r = resample_volume(normalize_01(raw_vol), TARGET_SHAPE)
    raw_r = np.clip(raw_r, 0, 1)

    mse_val  = float(sk_mse(raw_r, proc_vol))
    rmse_val = float(np.sqrt(mse_val))
    psnr_val = float(sk_psnr(raw_r, proc_vol, data_range=1.0)) if mse_val > 0 else float("inf")

    ssim_vals = []
    for i in range(raw_r.shape[2]):
        s1, s2 = raw_r[:, :, i], proc_vol[:, :, i]
        if s1.max() == s1.min() or s2.max() == s2.min():
            continue
        ssim_vals.append(sk_ssim(s1, s2, data_range=1.0))
    ssim_val = float(np.mean(ssim_vals)) if ssim_vals else float("nan")

    result = {"MSE": mse_val, "RMSE": rmse_val, "PSNR": psnr_val, "SSIM": ssim_val}

    # UQI / FSIM / VIF via sewar (2D, computed on a representative middle slice for speed)
    if HAVE_SEWAR:
        mid = raw_r.shape[2] // 2
        s1 = (raw_r[:, :, mid] * 255).astype(np.uint8)
        s2 = (proc_vol[:, :, mid] * 255).astype(np.uint8)
        try:
            result["UQI"] = float(sewar_uqi(s1, s2))
        except Exception:
            result["UQI"] = float("nan")
        try:
            result["FSIM"] = float(sewar_fsim(s1, s2))
        except Exception:
            result["FSIM"] = float("nan")
        try:
            result["VIF"] = float(sewar_vif(s1, s2))
        except Exception:
            result["VIF"] = float("nan")
    else:
        result["UQI"] = result["FSIM"] = result["VIF"] = float("nan")

    # GMSD (simple manual implementation, no extra dependency)
    try:
        mid = raw_r.shape[2] // 2
        s1 = raw_r[:, :, mid]
        s2 = proc_vol[:, :, mid]
        gx1, gy1 = np.gradient(s1)
        gx2, gy2 = np.gradient(s2)
        g1 = np.sqrt(gx1**2 + gy1**2)
        g2 = np.sqrt(gx2**2 + gy2**2)
        gms = (2 * g1 * g2 + 1e-4) / (g1**2 + g2**2 + 1e-4)
        result["GMSD"] = float(np.std(gms))
    except Exception:
        result["GMSD"] = float("nan")

    return result


def compute_noreference_metrics(proc_vol):
    \"\"\"BRISQUE / NIQE / PIQE via piq (if available) computed on the middle slice.\"\"\"
    result = {"BRISQUE": float("nan"), "NIQE": float("nan"), "PIQE": float("nan")}
    if not HAVE_PIQ:
        return result
    try:
        mid = proc_vol.shape[2] // 2
        sl = proc_vol[:, :, mid]
        t = torch.tensor(sl, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        t = t.clamp(0, 1)
        try:
            result["BRISQUE"] = float(piq.brisque(t, data_range=1.0))
        except Exception:
            pass
        try:
            result["NIQE"] = float(piq.niqe(t.repeat(1, 3, 1, 1), data_range=1.0))
        except Exception:
            pass
        # PIQE not always available in piq; skipped gracefully if absent
    except Exception:
        pass
    return result


## 7. Run the full pipeline over the dataset

For every discovered volume:
1. Load raw volume → compute RAW property metrics.
2. Preprocess (denoise, bias-correct, CLAHE, mask, resample) → compute PREPROCESSED property metrics.
3. Compute reference metrics (raw vs preprocessed) and no-reference metrics.
4. Save the preprocessed volume to `OUTPUT_ROOT` (mirrors input relative path).
5. Append two rows (stage=raw, stage=preprocessed) + one reference-metrics row to results.


In [ ]:

results = []

for _, row in tqdm(df_index.iterrows(), total=len(df_index), desc="Processing volumes"):
    fp = row["filepath"]
    try:
        raw_vol, affine, header = load_nii(fp)
        if raw_vol.ndim != 3:
            print(f"Skipping non-3D volume: {fp} shape={raw_vol.shape}")
            continue

        # --- RAW metrics ---
        raw_props = compute_property_metrics(raw_vol)

        # --- Preprocessing ---
        proc_vol = preprocess_volume(raw_vol)

        # --- PREPROCESSED metrics ---
        proc_props = compute_property_metrics(proc_vol)

        # --- Reference & no-reference metrics ---
        ref_metrics = compute_reference_metrics(raw_vol, proc_vol)
        nr_metrics  = compute_noreference_metrics(proc_vol)

        # --- Save preprocessed volume ---
        rel_path = os.path.relpath(fp, DATASET_ROOT)
        out_path = os.path.join(OUTPUT_ROOT, rel_path)
        if out_path.endswith(".nii"):
            out_path += ".gz"
        save_nii(proc_vol, affine, header, out_path)

        base_meta = {
            "patient_id": row["patient_id"],
            "region": row["region"],
            "condition": row["condition"],
            "modality": row["modality"],
            "source_filepath": fp,
            "output_filepath": out_path,
        }

        results.append({**base_meta, "stage": "raw", **raw_props})
        results.append({**base_meta, "stage": "preprocessed", **proc_props,
                         **ref_metrics, **nr_metrics})

    except Exception as e:
        print(f"FAILED on {fp}: {e}")

df_results = pd.DataFrame(results)
print(f"Total metric rows: {len(df_results)}")
df_results.head(20)


## 8. Save consolidated CSV

In [ ]:

df_results.to_csv(CSV_OUTPUT, index=False)
print(f"Saved metrics CSV to: {CSV_OUTPUT}")
df_results.describe(include="all").T


## 9. Quick visual sanity check

Displays raw vs preprocessed middle slice for a sample volume, and a summary comparison plot
of property metrics grouped by modality/stage.


In [ ]:

if len(df_index) > 0:
    sample = df_index.iloc[0]
    raw_vol, _, _ = load_nii(sample["filepath"])
    proc_vol = preprocess_volume(raw_vol)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(normalize_01(raw_vol)[:, :, raw_vol.shape[2] // 2], cmap="gray")
    axes[0].set_title(f"RAW — {sample['modality']} ({sample['patient_id']})")
    axes[0].axis("off")

    axes[1].imshow(proc_vol[:, :, proc_vol.shape[2] // 2], cmap="gray")
    axes[1].set_title("PREPROCESSED")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:

if len(df_results):
    metric_cols = ["mean", "std_dev", "contrast", "sharpness", "edge_strength",
                    "complexity", "entropy", "noise_level"]
    pivot = df_results.groupby(["modality", "stage"])[metric_cols].mean()
    display(pivot)

    pivot.reset_index().melt(id_vars=["modality", "stage"], value_vars=metric_cols) \
        .pivot_table(index="modality", columns="stage", values="value", aggfunc="mean")


## 10. Notes & next steps

- **Columns in the CSV:**
  `patient_id, region, condition, modality, source_filepath, output_filepath, stage`
  (`raw` / `preprocessed`) + property metrics (`mean, std_dev, contrast, sharpness,
  edge_strength, complexity, entropy, noise_level`) + on `preprocessed` rows only:
  reference metrics vs raw (`MSE, RMSE, PSNR, SSIM, UQI, FSIM, VIF, GMSD`) and no-reference
  metrics (`BRISQUE, NIQE`).
- **LPIPS / PIQE**: left out of the default run to keep this notebook CPU-friendly and
  dependency-light; `piq` (installed above) does support LPIPS-style scores if you want to
  extend `compute_reference_metrics` — happy to add it if needed.
- **This entire notebook runs on CPU.** GPU is not required until Stage 3 (AI-based MRI
  enhancement) and Stage 4 (ROI segmentation), where you'll train actual deep learning models.
- Update `DATASET_ROOT` once your Hackathon dataset is unzipped/mounted, then **Run All**.
